# CS 521 HW4, Problem 2: Certified Training for Multiple Norms (CURE)

Open this notebook in Google Colab and select a GPU runtime (*Runtime > Change runtime type > T4 GPU*).

Complete each part that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the verifier, the evaluation or the default hyperparameters.
Change them only if the homework tells you to.

**Threat model.** Let $f$ be the network and $x$ an image with the label $y$. The network is *certified* at $x$
for the norm $p$ if the verifier proves this: $f_y(x') - f_j(x') > 0$ for all $j \neq y$ and all $x'$ with
$\|x' - x\|_p \leq \epsilon_p$. For $\ell_\infty$, $x'$ must also be in $[0, 1]^{784}$.
We use $\epsilon_\infty = 0.1$ and $\epsilon_2 = 1.0$.
The *union certified accuracy* is the percentage of test images where the network is certified for the two norms.

This completed notebook covers **(a), (b), and (c)** and is kept local. The default starter data, architecture, attacks, verifier, training function and evaluator are unchanged. The saved numerical outputs were captured from the same definitions executed through the local harness on Apple M1 CPU with four threads, PyTorch 2.14.1 and Torchvision 0.29.2. Re-running this notebook on Colab regenerates the experiments; select a GPU as recommended by the starter, or use CPU. Local timing tables will differ on another machine. All nine part (a,b) runs use the full 60,000/10,000 MNIST split and the default 20 epochs. CURE variants each use seeds 0 and 1.

Part (c) adds four 20-epoch runs and a two-seed longer CURE-Max compute control. All work remains local. Recorded part (c) results come from the identical definitions executed through run_c.py.


In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

device = torch.device("cpu")
print("device:", device)
EPS = {"linf": 0.1, "l2": 1.0}
NORMS = ["linf", "l2"]


# Data and model

The data and the network are the same as in Problem 1. The network has 3 hidden layers with 50 neurons.
It has no normalization layer.

In [ ]:
def load_mnist(train):
    ds = torchvision.datasets.MNIST("data", train=train, download=True)
    x = ds.data.float().div(255).unsqueeze(1)
    return (x.to(device), ds.targets.to(device))


def batches(x, y, batch_size, shuffle):
    idx = (
        torch.randperm(len(x), device=x.device)
        if shuffle
        else torch.arange(len(x), device=x.device)
    )
    for i in range(0, len(x), batch_size):
        j = idx[i : i + batch_size]
        yield (x[j], y[j])


x_train, y_train = load_mnist(train=True)
x_test, y_test = load_mnist(train=False)
print(x_train.shape, x_test.shape)


def make_model():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 50),
        nn.ReLU(),
        nn.Linear(50, 50),
        nn.ReLU(),
        nn.Linear(50, 50),
        nn.ReLU(),
        nn.Linear(50, 10),
    ).to(device)


def linear_layers(model):
    return [m for m in model if isinstance(m, nn.Linear)]


# Attacks

`pgd(model, x, y, eps, norm, steps)` is an untargeted PGD attack with a random start.
Use `norm="linf"` or `norm="l2"`. The function returns the adversarial images of the restart with the highest loss.

In [ ]:
def _project(delta, eps, norm):
    if norm == "linf":
        return delta.clamp(-eps, eps)
    n = delta.flatten(1).norm(dim=1).clamp(min=1e-12).view(-1, 1, 1, 1)
    return delta * (eps / n).clamp(max=1.0)


def pgd(model, x, y, eps, norm, steps=10, alpha=None, restarts=1):
    alpha = alpha or 2.5 * eps / steps
    best_x, best_loss = (x.clone(), torch.full((len(x),), -float("inf"), device=x.device))
    for _ in range(restarts):
        if norm == "linf":
            delta = (torch.rand_like(x) * 2 - 1) * eps
        else:
            delta = torch.randn_like(x)
            r = torch.rand(len(x), 1, 1, 1, device=x.device) * eps
            delta = delta / delta.flatten(1).norm(dim=1).view(-1, 1, 1, 1) * r
        delta = (x + delta).clamp(0, 1) - x
        for _ in range(steps):
            delta.requires_grad_(True)
            loss = F.cross_entropy(model(x + delta), y)
            (grad,) = torch.autograd.grad(loss, delta)
            with torch.no_grad():
                if norm == "linf":
                    step = grad.sign()
                else:
                    step = grad / grad.flatten(1).norm(dim=1).clamp(min=1e-12).view(-1, 1, 1, 1)
                delta = _project(delta + alpha * step, eps, norm)
                delta = (x + delta).clamp(0, 1) - x
        with torch.no_grad():
            loss = F.cross_entropy(model(x + delta), y, reduction="none")
            better = loss > best_loss
            best_x[better], best_loss[better] = ((x + delta)[better], loss[better])
    return best_x.detach()


# Verifier

A *region* has a center `c` (a batch of images), a radius `r` and a norm.
For $\ell_\infty$, the region is the box $[c - r, c + r] \cap [0, 1]^{784}$.
For $\ell_2$, the region is the ball $\{x' : \|x' - c\|_2 \leq r\}$. We do not intersect the ball with $[0, 1]^{784}$.
This is sound, but the bounds are a little looser.

* `ibp_margin_bounds(model, c, r, norm, y)` is IBP from Problem 1. For $\ell_2$, the bounds of the first linear layer
  are exact: $W_i c + b_i \pm r \|W_i\|_2$ for each neuron $i$. The last linear layer is merged with the specification.
* `crown_margin_bounds(model, c, r, norm, y)` is CROWN. At the input, $\min_{x'} a^\top x'$ over the $\ell_2$ ball is
  $a^\top c - r \|a\|_2$.
* `certified(model, x, y, norm)` uses the larger of the two lower bounds.

The two bound functions return a tensor `m` with the shape `(batch, 10)`.
The value `m[:, j]` is a lower bound of $f_y - f_j$ on the region, and `m[:, y] = 0`. Do not change the verifier.

In [ ]:
def first_layer_bounds(layer, c, r, norm):
    c = c.flatten(1)
    if norm == "linf":
        l, u = ((c - r).clamp(0, 1), (c + r).clamp(0, 1))
        mid, rad = (layer((u + l) / 2), (u - l) / 2 @ layer.weight.abs().t())
    else:
        mid, rad = (layer(c), r * layer.weight.norm(dim=1))
    return (mid - rad, mid + rad)


def spec_matrix(y, n_classes=10):
    eye = torch.eye(n_classes, device=y.device)
    return eye[y].unsqueeze(1) - eye.unsqueeze(0)


def ibp_margin_bounds(model, c, r, norm, y):
    layers = linear_layers(model)
    l, u = first_layer_bounds(layers[0], c, r, norm)
    for layer in layers[1:-1]:
        l, u = (F.relu(l), F.relu(u))
        mid, rad = (layer((u + l) / 2), (u - l) / 2 @ layer.weight.abs().t())
        l, u = (mid - rad, mid + rad)
    l, u = (F.relu(l), F.relu(u))
    last = layers[-1]
    w = last.weight[y].unsqueeze(1) - last.weight.unsqueeze(0)
    b = last.bias[y].unsqueeze(1) - last.bias.unsqueeze(0)
    mid, rad = ((u + l) / 2, (u - l) / 2)
    return (w @ mid.unsqueeze(2)).squeeze(2) - (w.abs() @ rad.unsqueeze(2)).squeeze(2) + b


def _relu_relaxation(l, u):
    active, unstable = (l >= 0, (l < 0) & (u > 0))
    s = torch.where(active, torch.ones_like(l), torch.zeros_like(l))
    s = torch.where(unstable, u / (u - l).clamp(min=1e-12), s)
    c = torch.where(unstable, -s * l, torch.zeros_like(l))
    alpha = torch.where(active | unstable & (u > -l), torch.ones_like(l), torch.zeros_like(l))
    return (alpha, s, c)


def _backward_lower(A, layers, bounds, c0, r, norm):
    const = torch.zeros(A.shape[:2], device=A.device)
    for i in range(len(layers) - 1, -1, -1):
        const = const + A @ layers[i].bias
        A = A @ layers[i].weight
        if i > 0:
            alpha, s, c = _relu_relaxation(*bounds[i - 1])
            a_pos, a_neg = (A.clamp(min=0), A.clamp(max=0))
            const = const + (a_neg * c.unsqueeze(1)).sum(2)
            A = a_pos * alpha.unsqueeze(1) + a_neg * s.unsqueeze(1)
    c0 = c0.flatten(1)
    if norm == "linf":
        l0, u0 = ((c0 - r).clamp(0, 1), (c0 + r).clamp(0, 1))
        return (
            const
            + (A.clamp(min=0) * l0.unsqueeze(1)).sum(2)
            + (A.clamp(max=0) * u0.unsqueeze(1)).sum(2)
        )
    return const + (A @ c0.unsqueeze(2)).squeeze(2) - r * A.norm(dim=2)


def crown_margin_bounds(model, c, r, norm, y):
    layers = linear_layers(model)
    bounds = []
    for k in range(len(layers) - 1):
        eye = torch.eye(layers[k].out_features, device=c.device).expand(len(c), -1, -1)
        lo = _backward_lower(eye, layers[: k + 1], bounds, c, r, norm)
        hi = -_backward_lower(-eye, layers[: k + 1], bounds, c, r, norm)
        bounds.append((lo, hi))
    return _backward_lower(spec_matrix(y), layers, bounds, c, r, norm)


@torch.no_grad()
def certified(model, x, y, norm, eps=None):
    eps = EPS[norm] if eps is None else eps
    m = torch.maximum(
        ibp_margin_bounds(model, x, eps, norm, y), crown_margin_bounds(model, x, eps, norm, y)
    )
    m[torch.arange(len(y)), y] = float("inf")
    return m.min(dim=1).values > 0


# Evaluation

`evaluate(model, name)` uses the 10,000 test images. It returns the clean accuracy, the certified accuracy for each
norm and the union certified accuracy. With `attacks=True`, it also returns the PGD accuracy (50 steps) for each norm
and for the union. The PGD accuracy is an upper bound of the true robust accuracy.
Use `evaluate` for all the numbers in your report.

In [ ]:
def evaluate(model, name, attacks=False, n=10000, batch_size=1000):
    model.eval()
    counts = {k: 0 for k in ["clean", "linf", "l2", "union", "pgd_linf", "pgd_l2", "pgd_union"]}
    for x, y in batches(x_test[:n], y_test[:n], batch_size, False):
        with torch.no_grad():
            correct = model(x).argmax(1) == y
        cert = {p: certified(model, x, y, p) & correct for p in NORMS}
        counts["clean"] += correct.sum().item()
        counts["linf"] += cert["linf"].sum().item()
        counts["l2"] += cert["l2"].sum().item()
        counts["union"] += (cert["linf"] & cert["l2"]).sum().item()
        if attacks:
            rob = {}
            for p in NORMS:
                x_adv = pgd(model, x, y, EPS[p], p, steps=50)
                with torch.no_grad():
                    rob[p] = correct & (model(x_adv).argmax(1) == y)
                counts["pgd_" + p] += rob[p].sum().item()
            counts["pgd_union"] += (rob["linf"] & rob["l2"]).sum().item()
    res = {"name": name}
    res.update({k: 100 * v / n for k, v in counts.items() if attacks or not k.startswith("pgd")})
    return res


def print_results(rows):
    for r in rows:
        print(
            f"{r['name']:<26}"
            + "  ".join(
                (f"{k} {v:6.2f}" for k, v in r.items() if k != "time" and isinstance(v, float))
            )
            + (f"  time/epoch {r['time']:5.1f} s" if "time" in r else "")
        )


# Training loop

For each batch, `train(model, loss_fn)` calls `loss_fn(model, x, y, t)` and does one optimizer step.
The value `t` is the training progress in epochs (a float). The schedule is the same as in Problem 1:

* For $t < 1$, `eps_scale(t)` is 0 (standard training).
* For $1 \leq t < 11$, `eps_scale(t)` increases linearly from 0 to 1.
* For $t \geq 11$, `eps_scale(t)` is 1.

A loss function uses the radius `eps_scale(t) * EPS[norm]`.

If you give the argument `gp_beta`, `train` uses `gradient_projection` (part (b)) when the radius is at its
final value ($t \geq 11$).

In [ ]:
WARMUP, RAMP = (1, 10)
LR = 0.004


def eps_scale(t):
    return 0.0 if t <= WARMUP else min((t - WARMUP) / RAMP, 1.0)


def train(model, loss_fn, epochs=20, lr=LR, batch_size=512, seed=0, gp_beta=None, verbose=True):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[15], gamma=0.1)
    params = [p for p in model.parameters() if p.requires_grad]
    n_batches = (len(x_train) + batch_size - 1) // batch_size
    start = time.time()
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for i, (x, y) in enumerate(batches(x_train, y_train, batch_size, True)):
            t = epoch + i / n_batches
            loss = loss_fn(model, x, y, t)
            opt.zero_grad()
            if gp_beta is not None and t >= WARMUP + RAMP:
                g_c = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, gn, gc in zip(params, g_n, g_c):
                    p.grad = gradient_projection(gn, gc, gp_beta)
            else:
                loss.backward()
            opt.step()
            total += loss.item()
        sched.step()
        if verbose:
            print(
                f"epoch {epoch + 1:2d}  loss {total / n_batches:.4f}  time {time.time() - start:.0f} s"
            )
    return (time.time() - start) / epochs


def standard_loss(model, x, y, t):
    return F.cross_entropy(model(x), y)


# (a) Single-norm certified training

CURE uses the SABR loss from the lectures for each norm. SABR finds an adversarial example $x^*$ with PGD, and then it
calculates the IBP loss on a small ball of radius $\tau = \lambda \epsilon$ around $x^*$. PGD searches for $x^*$ in the
ball of radius $\epsilon - \tau$ around $x$, so that the small ball is inside the full ball.

1. `propagation_region`: Find a center $x^*$ with `pgd` (5 steps) in the ball of radius $\epsilon - \tau$ around $x$.
   Attack the model in eval mode. Then set the model back to train mode.
2. `certified_loss`: Calculate the cross-entropy of $\hat{z} = -\underline{m}$. Here, $\underline{m}$ are the IBP lower
   bounds of the margins on the small ball. Thus $\hat{z}_y = 0$.
   With `reduction="none"`, return the loss of each image.
3. `single_norm_loss`: For $t < 1$, return the standard loss. After that, return `certified_loss` with the radius
   `eps_scale(t) * EPS[norm]`.

The default ratios are `LAM = {"linf": 0.7, "l2": 0.7}`.

In [ ]:
LAM = {"linf": 0.7, "l2": 0.7}


def propagation_region(model, x, y, eps, lam, norm):
    tau = lam * eps
    model.eval()
    center = x.detach() if eps == 0 else pgd(model, x, y, eps - tau, norm, steps=5)
    model.train()
    return (center.detach(), tau)


def certified_loss(model, x, y, eps, lam, norm, reduction="mean"):
    center, tau = propagation_region(model, x, y, eps, lam, norm)
    z_hat = -ibp_margin_bounds(model, center, tau, norm, y)
    return F.cross_entropy(z_hat, y, reduction=reduction)


def single_norm_loss(model, x, y, t, norm):
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    return certified_loss(model, x, y, eps_scale(t) * EPS[norm], LAM[norm], norm)


In [ ]:
import json
from pathlib import Path
import time as _wall_time


class ClockRecorder:

    def __init__(self):
        self.calls = []

    def time(self):
        value = _wall_time.time()
        self.calls.append(value)
        return value


models, runs = ({}, [])
output = Path("results")
output.mkdir(exist_ok=True)


def run_method(name, seed, loss_fn, beta=None):
    torch.manual_seed(seed)
    model = make_model()
    print(f"BEGIN {name} seed={seed}")
    clock = ClockRecorder()
    real_time = globals()["time"]
    globals()["time"] = clock
    seconds = train(model, loss_fn, seed=seed, gp_beta=beta)
    globals()["time"] = real_time
    durations = [b - a for a, b in zip(clock.calls[:20], clock.calls[1:21])]
    torch.save(model.state_dict(), output / f"{name}_seed{seed}.pt")
    torch.manual_seed(10000 + seed)
    metric = evaluate(model, name, attacks=True)
    row = dict(
        name=name,
        seed=seed,
        complete=True,
        metrics=metric,
        mean_epoch_seconds=seconds,
        epoch_seconds=durations,
    )
    (output / f"{name}_seed{seed}.json").write_text(json.dumps(row, indent=2))
    models[name, seed] = model
    runs.append(row)
    print_results([dict(metric, time=seconds)])
    return row


for name, loss_fn in [
    ("standard", standard_loss),
    ("linf", lambda m, x, y, t: single_norm_loss(m, x, y, t, "linf")),
    ("l2", lambda m, x, y, t: single_norm_loss(m, x, y, t, "l2")),
]:
    run_method(name, 0, loss_fn)


BEGIN standard seed=0
epoch  1  loss 0.5741  time 0 s
epoch  2  loss 0.2118  time 0 s
epoch  3  loss 0.1582  time 0 s
epoch  4  loss 0.1259  time 1 s
epoch  5  loss 0.1051  time 1 s
epoch  6  loss 0.0906  time 1 s
epoch  7  loss 0.0798  time 1 s
epoch  8  loss 0.0697  time 1 s
epoch  9  loss 0.0626  time 1 s
epoch 10  loss 0.0582  time 2 s
epoch 11  loss 0.0499  time 2 s
epoch 12  loss 0.0463  time 2 s
epoch 13  loss 0.0449  time 2 s
epoch 14  loss 0.0379  time 2 s
epoch 15  loss 0.0347  time 2 s
epoch 16  loss 0.0190  time 2 s
epoch 17  loss 0.0147  time 3 s
epoch 18  loss 0.0134  time 3 s
epoch 19  loss 0.0125  time 3 s
epoch 20  loss 0.0116  time 3 s
{"name": "standard", "clean": 97.23, "linf": 0.0, "l2": 0.0, "union": 0.0, "pgd_linf": 1.81, "pgd_l2": 11.73, "pgd_union": 1.78}
BEGIN linf seed=0
epoch  1  loss 0.5741  time 0 s
epoch  2  loss 0.4888  time 3 s
epoch  3  loss 0.4810  time 5 s
epoch  4  loss 0.4858  time 8 s
epoch  5  loss 0.5131  time 11 s
epoch  6  loss 0.5536  time 13

# (b) CURE

Let $\hat{z}^\infty$ and $\hat{z}^2$ be the negative IBP margin bounds on the two small balls of an image.
These are the same values as in `certified_loss`. Let $d^p = \mathrm{softmax}(\hat{z}^p)$. Complete `cure_loss`:
$$\mathcal{L}_{CURE} = \frac{1}{n} \sum_{i=1}^{n} \max\left(\mathcal{L}_{CE}(\hat{z}^\infty_i, y_i),\ \mathcal{L}_{CE}(\hat{z}^2_i, y_i)\right)
+ \eta \cdot \frac{1}{|\gamma|} \sum_{i \in \gamma} \mathrm{KL}\left(d^2_i \,\|\, d^\infty_i\right),$$
Here, $\gamma$ is the set of images in the batch that are certified for $\ell_2$ on their small ball.
For these images, all entries of $\hat{z}^2_i$ except $\hat{z}^2_{i, y_i}$ are negative.
The first term is CURE-Max. The second term is the bound alignment (BA).
For $t < 1$, return the standard loss. If $\gamma$ is empty, the second term is 0.

Then complete `gradient_projection` (GP). Let $g_c$ be the gradient of `cure_loss`, and $g_n$ the gradient of the
cross-entropy loss on the clean images. For each parameter tensor:
$$g = \beta \cdot \mathrm{GP}(g_n, g_c) + (1 - \beta) \cdot g_c, \qquad
\mathrm{GP}(g_n, g_c) = \begin{cases} \cos(g_n, g_c) \cdot g_n & \text{if } \cos(g_n, g_c) > 0 \\ 0 & \text{otherwise.} \end{cases}$$
This is the same operation as in RAMP.

This is a simplified version of CURE. The paper is different in three ways:

* It propagates an $\ell_\infty$ box also for $\ell_2$.
* It uses a different $\ell_2$ attack.
* It applies GP to the updates of a full epoch.

In [ ]:
def cure_loss(model, x, y, t, eta=0.5):
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    z = {}
    for norm in ["linf", "l2"]:
        center, tau = propagation_region(model, x, y, eps_scale(t) * EPS[norm], LAM[norm], norm)
        z[norm] = -ibp_margin_bounds(model, center, tau, norm, y)
    ce_inf = F.cross_entropy(z["linf"], y, reduction="none")
    ce_two = F.cross_entropy(z["l2"], y, reduction="none")
    loss = torch.maximum(ce_inf, ce_two).mean()
    if eta != 0:
        competitors = ~F.one_hot(y, num_classes=z["l2"].shape[1]).bool()
        gamma = z["l2"].masked_fill(~competitors, -float("inf")).max(1).values < 0
        if gamma.any():
            log_inf = F.log_softmax(z["linf"][gamma], dim=1)
            log_two = F.log_softmax(z["l2"][gamma], dim=1)
            alignment = F.kl_div(log_inf, log_two, reduction="none", log_target=True).sum(1).mean()
            loss = loss + eta * alignment
    return loss


def gradient_projection(g_n, g_c, beta):
    dot = (g_n * g_c).sum()
    norm_product = g_n.norm() * g_c.norm()
    cosine = dot / norm_product.clamp_min(1e-12)
    projected = cosine.clamp_min(0) * g_n
    return beta * projected + (1 - beta) * g_c


In [ ]:
for name, loss_fn, beta in [
    ("cure_max", lambda m, x, y, t: cure_loss(m, x, y, t, eta=0.0), None),
    ("cure_max_ba", cure_loss, None),
    ("cure_max_ba_gp", cure_loss, 0.5),
]:
    for seed in [0, 1]:
        run_method(name, seed, loss_fn, beta)
Path("results/results.json").write_text(json.dumps(dict(complete=True, runs=runs), indent=2))


BEGIN cure_max seed=0
epoch  1  loss 0.5741  time 0 s
epoch  2  loss 0.4905  time 6 s
epoch  3  loss 0.5029  time 11 s
epoch  4  loss 0.5366  time 16 s
epoch  5  loss 0.5945  time 22 s
epoch  6  loss 0.6691  time 27 s
epoch  7  loss 0.7523  time 33 s
epoch  8  loss 0.8534  time 38 s
epoch  9  loss 0.9706  time 43 s
epoch 10  loss 1.1024  time 49 s
epoch 11  loss 1.2429  time 54 s
epoch 12  loss 1.2939  time 60 s
epoch 13  loss 1.2686  time 65 s
epoch 14  loss 1.2510  time 70 s
epoch 15  loss 1.2382  time 76 s
epoch 16  loss 1.1960  time 81 s
epoch 17  loss 1.1833  time 87 s
epoch 18  loss 1.1805  time 92 s
epoch 19  loss 1.1810  time 97 s
epoch 20  loss 1.1783  time 103 s
{"name": "cure_max", "clean": 92.27, "linf": 67.18, "l2": 56.91, "union": 56.77, "pgd_linf": 76.52, "pgd_l2": 78.01, "pgd_union": 75.89}
BEGIN cure_max seed=1
epoch  1  loss 0.5738  time 0 s
epoch  2  loss 0.4906  time 6 s
epoch  3  loss 0.5041  time 11 s
epoch  4  loss 0.5446  time 16 s
epoch  5  loss 0.5934  time 22

## Recorded results

CURE rows are two-seed means; all other rows use seed 0. Every accuracy uses the original evaluator.

| Method | Clean | Cert. $\infty$ | Cert. 2 | Cert. union | PGD $\infty$ | PGD 2 | PGD union |
|----------------------|-------:|-------:|-------:|-------:|-------:|-------:|-------:|
| Standard | 97.23 | 0.00 | 0.00 | 0.00 | 1.81 | 11.73 | 1.78 |
| $\ell_\infty$ SABR | 93.03 | 80.61 | 5.19 | 5.19 | 84.00 | 69.73 | 69.52 |
| $\ell_2$ SABR | 92.69 | 54.59 | 58.88 | 51.25 | 68.59 | 78.05 | 68.08 |
| CURE-Max | 92.22 | 67.45 | 57.44 | 57.33 | 76.86 | 78.48 | 76.32 |
| CURE + BA | 92.36 | 67.37 | 57.30 | 57.22 | 76.36 | 78.22 | 75.88 |
| CURE + BA + GP | 92.58 | 66.91 | 56.78 | 56.67 | 76.31 | 78.31 | 75.88 |

| Method | Seeds | Mean seconds / epoch | Mean 20-epoch training time |
|---|---|---:|---:|
| Standard | 0 | 0.151 | 3.01 s |
| $\ell_\infty$ SABR | 0 | 2.526 | 50.51 s |
| $\ell_2$ SABR | 0 | 2.818 | 56.35 s |
| CURE-Max | 0, 1 | 5.144 | 102.89 s |
| CURE + BA | 0, 1 | 5.168 | 103.35 s |
| CURE + BA + GP | 0, 1 | 5.269 | 105.38 s |

| CURE method | Seed | Clean | Cert. $\infty$ | Cert. 2 | Cert. union | PGD union |
|---|---:|---:|---:|---:|---:|---:|
| CURE-Max | 0 | 92.27 | 67.18 | 56.91 | 56.77 | 75.89 |
| CURE-Max | 1 | 92.16 | 67.72 | 57.98 | 57.89 | 76.75 |
| CURE + BA | 0 | 92.82 | 67.24 | 57.21 | 57.09 | 75.77 |
| CURE + BA | 1 | 91.90 | 67.50 | 57.40 | 57.34 | 75.98 |
| CURE + BA + GP | 0 | 93.05 | 66.86 | 56.84 | 56.73 | 75.85 |
| CURE + BA + GP | 1 | 92.11 | 66.96 | 56.71 | 56.61 | 75.92 |

## Part (a) analysis

**(a)(i) Transfer across norms.** The $\ell_\infty$ model certifies 80.61% at its training norm but only 5.19% at $\ell_2$. Yet its $\ell_2$ PGD accuracy is 69.73%, showing empirical transfer that the full-ball verifier cannot establish. This gap can reflect loose bounds and the starter's stronger, unclipped $\ell_2$ verification region, while PGD clips pixels. The $\ell_2$ model transfers more certification: 58.88% at $\ell_2$ and 54.59% at $\ell_\infty$, with PGD accuracies 78.05% and 68.59%. Both models improve cross-norm PGD accuracy over standard training. The threat sets are not nested: $\|\delta\|_\infty\leq0.1$ permits $\|\delta\|_2$ as large as 2.8 in 784 dimensions, while $\|\delta\|_2\leq1$ permits coordinates larger than 0.1. Transfer is therefore not guaranteed.

**(a)(ii) Limiting norm.** Standard training certifies 0% for both norms. For $\ell_\infty$ SABR, $\ell_2$ is the bottleneck: both its $\ell_2$ and union certificates are 5.19%. For $\ell_2$ SABR, $\ell_\infty$ has the lower marginal certificate (54.59% versus 58.88%). Its union is lower still (51.25%), because the images certified for the two norms do not form nested sets.

## Part (b) analysis

**(b)(i) Comparison.** CURE-Max has the highest union certified accuracy, 57.33%; Standard has the highest clean accuracy, 97.23%. The strongest single-norm union certificate is 51.25%. CURE-Max achieves 57.33%, a change of +6.08 points. Its per-image maximum explicitly trains against whichever norm currently has the higher certified loss.

**(b)(ii) BA and GP.** Adding BA decreases mean union certification by 0.11 points. Paired seed-0/seed-1 changes are +0.32/-0.55; the marginal certificates change by -0.08 ($\ell_\infty$) and -0.14 ($\ell_2$) points. BA aligns bound distributions only on the small-ball $\ell_2$-certified subset, so it does not guarantee better full-ball union certificates. Adding GP to BA decreases union certification by 0.55 points, with paired changes -0.36/-0.73. Its clean accuracy changes by +0.22 points, and its marginal certificates by -0.46/-0.53. GP favors clean-gradient components aligned with the certified gradient; this changes the clean/robust balance rather than directly maximizing the number of certificates. These per-norm changes show how the balance shifts in our runs. Two seeds establish observed differences, not statistical significance.

**(b)(iii) Cost.** CURE-Max takes 5.144 s/epoch: 2.04 times $\ell_\infty$ SABR and 1.83 times $\ell_2$ SABR. BA uses 5.168 s/epoch, and BA+GP uses 5.269 s/epoch. CURE computes two five-step PGD centers and two interval losses per batch; BA adds the distribution alignment, while GP adds a separate clean-loss gradient after the radius reaches its final value. CPU timings can vary with system load.

## Per-epoch times (seconds)

| Epoch | Standard | $\ell_\infty$ SABR | $\ell_2$ SABR |
|---|---:|---:|---:|
| 1 | 0.200 | 0.172 | 0.173 |
| 2 | 0.149 | 2.579 | 2.888 |
| 3 | 0.147 | 2.649 | 2.918 |
| 4 | 0.146 | 2.620 | 2.896 |
| 5 | 0.151 | 2.623 | 2.916 |
| 6 | 0.150 | 2.624 | 2.902 |
| 7 | 0.146 | 2.629 | 2.911 |
| 8 | 0.148 | 2.629 | 2.897 |
| 9 | 0.146 | 2.621 | 3.252 |
| 10 | 0.149 | 2.721 | 2.939 |
| 11 | 0.146 | 2.850 | 2.935 |
| 12 | 0.150 | 2.608 | 2.993 |
| 13 | 0.154 | 2.828 | 3.004 |
| 14 | 0.147 | 2.726 | 3.168 |
| 15 | 0.148 | 2.603 | 2.901 |
| 16 | 0.146 | 2.594 | 2.932 |
| 17 | 0.151 | 2.605 | 2.975 |
| 18 | 0.146 | 2.609 | 2.957 |
| 19 | 0.147 | 2.608 | 2.899 |
| 20 | 0.148 | 2.618 | 2.896 |

| Epoch | Max, s0 | Max, s1 | BA, s0 | BA, s1 | BA+GP, s0 | BA+GP, s1 |
|---|---:|---:|---:|---:|---:|---:|
| 1 | 0.168 | 0.179 | 0.169 | 0.168 | 0.164 | 0.168 |
| 2 | 5.348 | 5.386 | 5.374 | 5.365 | 5.454 | 5.396 |
| 3 | 5.404 | 5.370 | 5.461 | 5.393 | 5.428 | 5.422 |
| 4 | 5.411 | 5.401 | 5.372 | 5.492 | 5.459 | 5.445 |
| 5 | 5.429 | 5.455 | 5.375 | 5.431 | 5.445 | 5.455 |
| 6 | 5.414 | 5.439 | 5.368 | 5.458 | 5.412 | 5.439 |
| 7 | 5.377 | 5.503 | 5.370 | 5.435 | 5.609 | 5.470 |
| 8 | 5.401 | 5.461 | 5.425 | 5.437 | 5.542 | 5.474 |
| 9 | 5.409 | 5.368 | 5.433 | 5.433 | 5.475 | 5.716 |
| 10 | 5.393 | 5.329 | 5.403 | 5.512 | 5.358 | 5.440 |
| 11 | 5.389 | 5.311 | 5.413 | 5.468 | 5.456 | 5.420 |
| 12 | 5.405 | 5.311 | 5.395 | 5.428 | 5.583 | 5.632 |
| 13 | 5.383 | 5.351 | 5.463 | 5.499 | 5.578 | 5.592 |
| 14 | 5.394 | 5.387 | 5.432 | 5.477 | 5.654 | 5.683 |
| 15 | 5.429 | 5.450 | 5.431 | 5.489 | 5.681 | 5.578 |
| 16 | 5.380 | 5.351 | 5.394 | 5.454 | 5.585 | 5.589 |
| 17 | 5.376 | 5.790 | 5.386 | 5.435 | 5.620 | 5.594 |
| 18 | 5.371 | 5.617 | 5.442 | 5.423 | 5.647 | 5.590 |
| 19 | 5.492 | 5.337 | 5.474 | 5.480 | 5.594 | 5.690 |
| 20 | 5.374 | 5.226 | 5.389 | 5.460 | 5.597 | 5.621 |


In [ ]:
from IPython.display import display, Markdown

keys = ["clean", "linf", "l2", "union", "pgd_linf", "pgd_l2", "pgd_union"]
lines = ["| Method | " + " | ".join(keys) + " | s/epoch |", "|---|" + "---:|" * 8]
for name in dict.fromkeys((row["name"] for row in runs)):
    rr = [row for row in runs if row["name"] == name]
    values = [sum((row["metrics"][key] for row in rr)) / len(rr) for key in keys]
    mean_time = sum((row["mean_epoch_seconds"] for row in rr)) / len(rr)
    lines.append(
        "| " + name + " | " + " | ".join((f"{v:.2f}" for v in values)) + f" | {mean_time:.3f} |"
    )
display(Markdown("\n".join(lines)))
for row in runs:
    print(row["name"], row["seed"], [round(t, 3) for t in row["epoch_seconds"]])


# (c) Bound-aware region search

 Idea and justification

The best part (b) method is CURE-Max. Its union certificate is nearly its $\ell_2$ certificate, suggesting that $\ell_2$ is the main bottleneck. SABR selects a small propagation region around an adversarial center [1], but the course implementation finds that center using concrete cross-entropy. I instead select the center using the loss of the entire small IBP region. A point with high concrete loss need not produce the most difficult interval bound.

For each norm $p$, let $\epsilon_p(t)$ be the unchanged scheduled radius and $\tau_p=0.7\epsilon_p(t)$. Define

$$
J_p(c)=\operatorname{CE}(-\underline m_{\mathrm{IBP}}(c,\tau_p,p),y),\qquad
c_p^*\approx\arg\max_{\|c-x\|_p\leq\epsilon_p(t)-\tau_p,\;c\in[0,1]^d}J_p(c).
$$

The new training loss is the per-image maximum of the resulting two region losses:

$$
L_{\mathrm{new}}=\frac1n\sum_i\max\{J_\infty(c_{i,\infty}^*),J_2(c_{i,2}^*)\}.
$$

This extends CURE-Max [2] by changing the inner search objective, not merely a hyperparameter. The proposal is an adaptation of SABR/CURE to this assignment; I do not claim that bound-aware region search is globally new. Selecting harder regions is a hypothesis for better full-ball certification, not a guarantee: small-region training still does not certify the full threat set.



In [ ]:
def bound_pgd(model, x, y, radius, tau, norm, steps=5):
    alpha = 2.5 * radius / steps
    if norm == "linf":
        delta = (torch.rand_like(x) * 2 - 1) * radius
    else:
        delta = torch.randn_like(x)
        r = torch.rand(len(x), 1, 1, 1, device=x.device) * radius
        delta = delta / delta.flatten(1).norm(dim=1).view(-1, 1, 1, 1) * r
    delta = (x + delta).clamp(0, 1) - x
    for _ in range(steps):
        delta = delta.detach().requires_grad_(True)
        logits = -ibp_margin_bounds(model, x + delta, tau, norm, y)
        loss = F.cross_entropy(logits, y)
        (grad,) = torch.autograd.grad(loss, delta)
        with torch.no_grad():
            if norm == "linf":
                direction = grad.sign()
            else:
                direction = grad / grad.flatten(1).norm(dim=1).clamp_min(1e-12).view(-1, 1, 1, 1)
            delta = _project(delta + alpha * direction, radius, norm)
            delta = (x + delta).clamp(0, 1) - x
    return (x + delta).detach()


def bound_aware_region(model, x, y, eps, lam, norm):
    tau = lam * eps
    model.eval()
    center = x.detach() if eps == 0 else bound_pgd(model, x, y, eps - tau, tau, norm)
    model.train()
    return (center, tau)


def region_max_loss(model, x, y, t, bound_norms):
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    losses = []
    for norm in ["linf", "l2"]:
        region = bound_aware_region if norm in bound_norms else propagation_region
        center, tau = region(model, x, y, eps_scale(t) * EPS[norm], LAM[norm], norm)
        logits = -ibp_margin_bounds(model, center, tau, norm, y)
        losses.append(F.cross_entropy(logits, y, reduction="none"))
    return torch.maximum(*losses).mean()


def my_loss(model, x, y, t):
    return region_max_loss(model, x, y, t, bound_norms=("linf", "l2"))


def ablation_loss(model, x, y, t):
    return region_max_loss(model, x, y, t, bound_norms=("l2",))


In [ ]:
import math

c_runs, c_models = ([], {})
c_output = Path("results_c")
c_output.mkdir(exist_ok=True)


def run_c_method(name, seed, loss_fn, epochs=20):
    torch.manual_seed(seed)
    model = make_model()
    print(f"BEGIN {name} seed={seed} epochs={epochs}")
    clock = ClockRecorder()
    real_time = globals()["time"]
    globals()["time"] = clock
    seconds = train(model, loss_fn, epochs=epochs, seed=seed)
    globals()["time"] = real_time
    durations = [b - a for a, b in zip(clock.calls[:epochs], clock.calls[1 : epochs + 1])]
    torch.save(model.state_dict(), c_output / f"{name}_seed{seed}.pt")
    torch.manual_seed(10000 + seed)
    metrics = evaluate(model, name, attacks=True)
    row = dict(
        name=name,
        seed=seed,
        complete=True,
        metrics=metrics,
        config=dict(epochs=epochs),
        mean_epoch_seconds=seconds,
        total_training_seconds=epochs * seconds,
        epoch_seconds=durations,
    )
    (c_output / f"{name}_seed{seed}.json").write_text(json.dumps(row, indent=2))
    c_models[name, seed] = model
    c_runs.append(row)
    print_results([dict(metrics, time=seconds)])
    return row


for name, loss_fn in [("bound_aware", my_loss), ("l2_only", ablation_loss)]:
    for seed in [0, 1]:
        run_c_method(name, seed, loss_fn)


BEGIN bound_aware seed=0 epochs=20
epoch  1  loss 0.5741  time 0 s
epoch  2  loss 0.5022  time 9 s
epoch  3  loss 0.5088  time 19 s
epoch  4  loss 0.5430  time 28 s
epoch  5  loss 0.6066  time 37 s
epoch  6  loss 0.6904  time 46 s
epoch  7  loss 0.7886  time 56 s
epoch  8  loss 0.9053  time 66 s
epoch  9  loss 1.0326  time 75 s
epoch 10  loss 1.1755  time 84 s
epoch 11  loss 1.3227  time 94 s
epoch 12  loss 1.3768  time 103 s
epoch 13  loss 1.3535  time 113 s
epoch 14  loss 1.3381  time 122 s
epoch 15  loss 1.3227  time 131 s
epoch 16  loss 1.2825  time 141 s
epoch 17  loss 1.2723  time 150 s
epoch 18  loss 1.2700  time 159 s
epoch 19  loss 1.2694  time 169 s
epoch 20  loss 1.2683  time 178 s
{"name": "bound_aware", "clean": 92.05, "linf": 66.92, "l2": 57.18, "union": 57.13, "pgd_linf": 76.27, "pgd_l2": 77.46, "pgd_union": 75.45}
BEGIN bound_aware seed=1 epochs=20
epoch  1  loss 0.5738  time 0 s
epoch  2  loss 0.5065  time 10 s
epoch  3  loss 0.5109  time 20 s
epoch  4  loss 0.5506  ti

In [ ]:
baseline_runs = [row for row in runs if row["name"] == "cure_max"]
target = max((row["total_training_seconds"] for row in c_runs if row["name"] == "bound_aware"))
rate = sum((row["mean_epoch_seconds"] for row in baseline_runs)) / 2
control_epochs = max(20, math.ceil(1.05 * target / rate))
for seed in [0, 1]:
    run_c_method(
        "cure_time_control",
        seed,
        lambda m, x, y, t: cure_loss(m, x, y, t, eta=0.0),
        epochs=control_epochs,
    )
(c_output / "results.json").write_text(
    json.dumps(dict(complete=True, runs=c_runs, baseline_runs=baseline_runs), indent=2)
)


BEGIN cure_time_control seed=0 epochs=38
epoch  1  loss 0.5741  time 0 s
epoch  2  loss 0.4905  time 5 s
epoch  3  loss 0.5029  time 10 s
epoch  4  loss 0.5366  time 15 s
epoch  5  loss 0.5945  time 20 s
epoch  6  loss 0.6691  time 25 s
epoch  7  loss 0.7523  time 30 s
epoch  8  loss 0.8534  time 35 s
epoch  9  loss 0.9706  time 40 s
epoch 10  loss 1.1024  time 45 s
epoch 11  loss 1.2429  time 50 s
epoch 12  loss 1.2939  time 56 s
epoch 13  loss 1.2686  time 61 s
epoch 14  loss 1.2510  time 66 s
epoch 15  loss 1.2382  time 71 s
epoch 16  loss 1.1960  time 76 s
epoch 17  loss 1.1833  time 81 s
epoch 18  loss 1.1805  time 86 s
epoch 19  loss 1.1810  time 91 s
epoch 20  loss 1.1783  time 96 s
epoch 21  loss 1.1740  time 101 s
epoch 22  loss 1.1748  time 106 s
epoch 23  loss 1.1724  time 111 s
epoch 24  loss 1.1718  time 116 s
epoch 25  loss 1.1678  time 121 s
epoch 26  loss 1.1688  time 126 s
epoch 27  loss 1.1651  time 131 s
epoch 28  loss 1.1664  time 136 s
epoch 29  loss 1.1639  time 1

In [ ]:
rows = []
for seed in [0, 1]:
    model = models["cure_max", seed]
    model.eval()
    x, y = (x_train[:512], y_train[:512])
    for k, norm in enumerate(NORMS):
        random_seed = 20260 + seed * 10 + k
        torch.manual_seed(random_seed)
        center, tau = propagation_region(model, x, y, EPS[norm], 0.7, norm)
        with torch.no_grad():
            old = F.cross_entropy(
                -ibp_margin_bounds(model, center, tau, norm, y), y, reduction="none"
            )
        torch.manual_seed(random_seed)
        center, tau = bound_aware_region(model, x, y, EPS[norm], 0.7, norm)
        with torch.no_grad():
            new = F.cross_entropy(
                -ibp_margin_bounds(model, center, tau, norm, y), y, reduction="none"
            )
        row = dict(
            seed=seed,
            norm=norm,
            samples=len(x),
            ce_center_bound_loss=float(old.mean()),
            bound_center_bound_loss=float(new.mean()),
            mean_increase=float((new - old).mean()),
            fraction_increased=float((new - old > 1e-06).float().mean()),
            random_seed=random_seed,
        )
        rows.append(row)
        print(row, flush=True)
(c_output / "center_diagnostic.json").write_text(
    json.dumps(
        dict(
            description="Fixed baseline CURE-Max checkpoints, first 512 training images, matched random starts and five search steps. Training-objective diagnostic, not an accuracy evaluation.",
            rows=rows,
        ),
        indent=2,
    )
)


{"seed": 0, "norm": "linf", "samples": 512, "ce_center_bound_loss": 0.9717167615890503, "bound_center_bound_loss": 1.0501079559326172, "mean_increase": 0.07839112728834152, "fraction_increased": 0.99609375, "random_seed": 20260}
{"seed": 0, "norm": "l2", "samples": 512, "ce_center_bound_loss": 1.0488396883010864, "bound_center_bound_loss": 1.1638448238372803, "mean_increase": 0.11500513553619385, "fraction_increased": 0.9921875, "random_seed": 20261}
{"seed": 1, "norm": "linf", "samples": 512, "ce_center_bound_loss": 0.9764171838760376, "bound_center_bound_loss": 1.0476658344268799, "mean_increase": 0.07124868035316467, "fraction_increased": 0.994140625, "random_seed": 20270}
{"seed": 1, "norm": "l2", "samples": 512, "ce_center_bound_loss": 1.045257568359375, "bound_center_bound_loss": 1.1455222368240356, "mean_increase": 0.10026471316814423, "fraction_increased": 0.958984375, "random_seed": 20271}


## Recorded part (c) results and analysis

| Method | Clean | Cert. $\infty$ | Cert. 2 | Cert. union | PGD $\infty$ | PGD 2 | PGD union |
|----------------------|-------:|-------:|-------:|-------:|-------:|-------:|-------:|
| CURE-Max | 92.22 | 67.45 | 57.44 | 57.33 | 76.86 | 78.48 | 76.32 |
| Bound-aware, both | 91.73 | 66.80 | 57.75 | 57.65 | 75.41 | 76.92 | 74.78 |
| Bound-aware, L2 only | 92.04 | 66.28 | 58.20 | 58.00 | 75.89 | 77.68 | 75.20 |
| Longer CURE-Max | 92.25 | 67.78 | 57.61 | 57.50 | 77.65 | 79.19 | 77.04 |

| Method | Epochs | Mean seconds / epoch | Mean total training seconds |
|---|---:|---:|---:|
| CURE-Max | 20 | 5.144 | 102.89 |
| Bound-aware, both | 20 | 9.094 | 181.88 |
| Bound-aware, L2 only | 20 | 6.175 | 123.49 |
| Longer CURE-Max | 38 | 4.926 | 187.18 |

| Method | Seed | Clean | Cert. union | PGD union | Total training (s) |
|---|---:|---:|---:|---:|---:|
| CURE-Max | 0 | 92.27 | 56.77 | 75.89 | 102.75 |
| CURE-Max | 1 | 92.16 | 57.89 | 76.75 | 103.02 |
| Bound-aware, both | 0 | 92.05 | 57.13 | 75.45 | 178.22 |
| Bound-aware, both | 1 | 91.42 | 58.17 | 74.10 | 185.53 |
| Bound-aware, L2 only | 0 | 92.32 | 57.71 | 75.47 | 122.78 |
| Bound-aware, L2 only | 1 | 91.77 | 58.28 | 74.94 | 124.21 |
| Longer CURE-Max | 0 | 92.35 | 56.83 | 76.50 | 185.97 |
| Longer CURE-Max | 1 | 92.14 | 58.17 | 77.58 | 188.40 |



| Method | Epochs | Mean seconds / epoch | Mean total training seconds |
|---|---:|---:|---:|
| CURE-Max | 20 | 5.144 | 102.89 |
| Bound-aware, both | 20 | 9.094 | 181.88 |
| Bound-aware, L2 only | 20 | 6.175 | 123.49 |
| Longer CURE-Max | 38 | 4.926 | 187.18 |

**20-epoch comparison.** The new method improves the mean 20-epoch union certificate: 57.33% to 57.65% (+0.32 points). Paired seed-0/seed-1 changes are +0.36/+0.28 points. Clean accuracy changes from 92.22% to 91.73% (-0.48 points). The marginal certificate changes are -0.65 points for $\ell_\infty$ and +0.31 for $\ell_2$. These are observed two-seed differences, not a claim of statistical significance.

Certification rises while clean accuracy and union PGD accuracy fall (PGD change -1.54 points). This is consistent with stronger verification-oriented regularization, not a general improvement in empirical robustness.

**Ablation.** Removing bound-aware search from $\ell_\infty$ gives 58.00% union certification, versus 57.65% for the full method (+0.34 points), with clean accuracy 92.04% versus 91.73%. Thus applying the new search to $\ell_\infty$ does not help in these runs. This isolates whether improving the limiting norm alone suffices. An empty set of bound-aware norms exactly recovers CURE-Max; that recovery is also tested numerically. The two-norm variant was the main method before looking at these results; the ablation is not relabeled post hoc as the original proposal.

**Compute control.** The main method takes 1.77 times the original baseline's training time. The longer baseline trains from scratch for 38 epochs, retaining the original learning-rate milestone and radius schedule; extra epochs use the final radius and decayed learning rate. Its mean training time is 187.18 s versus 181.88 s for the new method (+2.9%). This is an approximately time-matched, deliberately generous baseline budget, with exact times reported rather than claimed equal. The new method also exceeds its 57.50% union certificate (+0.15 points). Extra compute and the changed objective are therefore assessed separately.

**What the search changes.** On the first 512 training images, with fixed original CURE-Max checkpoints and matched random starts, I compare the same small-region loss at concrete-CE and bound-aware centers. These are training-objective diagnostics, not extra accuracy measurements:

| Baseline seed | Norm | CE-center region loss | Bound-center region loss | Images with larger region loss |
|---|---|---:|---:|---:|
| 0 | linf | 0.9717 | 1.0501 | 99.61% |
| 0 | l2 | 1.0488 | 1.1638 | 99.22% |
| 1 | linf | 0.9764 | 1.0477 | 99.41% |
| 1 | l2 | 1.0453 | 1.1455 | 95.90% |

The bound-aware search finds higher region losses for most of these images under both norms. This supports the inner-optimization premise, but does not by itself establish better robustness. The new model's union PGD accuracy is 74.78% versus 57.65% certified accuracy, a 17.12-point gap. Remaining gaps may reflect bound looseness, imperfect attack search, and the starter's unclipped $\ell_2$ verification ball versus pixel-clipped PGD. Harder bound objectives can also over-regularize the network or optimize relaxation artifacts, so the clean accuracy, per-norm certificates, ablation, and compute control are needed alongside this diagnostic.

**Checks.** Tests verify region containment, clipping, detached centers, finite outer gradients, mode restoration, no search-time parameter-gradient accumulation, interior zero-radius equivalence to concrete PGD, and exact recovery of the CURE-Max loss when the modification is removed. Saved metrics, checkpoints, seeds, default settings, protected-function hashes, and timing sums are audited.


## Per-epoch method/ablation times

| Epoch | Both, seed 0 | Both, seed 1 | L2 only, seed 0 | L2 only, seed 1 |
|---|---:|---:|---:|---:|
| 1 | 0.170 | 0.170 | 0.165 | 0.172 |
| 2 | 9.150 | 9.628 | 6.653 | 6.436 |
| 3 | 9.290 | 9.752 | 6.416 | 6.545 |
| 4 | 9.257 | 9.736 | 6.775 | 6.762 |
| 5 | 9.288 | 9.819 | 6.588 | 6.558 |
| 6 | 9.294 | 9.862 | 6.431 | 6.507 |
| 7 | 9.853 | 9.800 | 6.516 | 6.513 |
| 8 | 9.392 | 9.698 | 6.408 | 6.562 |
| 9 | 9.355 | 9.649 | 6.406 | 6.522 |
| 10 | 9.446 | 9.723 | 6.468 | 6.517 |
| 11 | 9.365 | 9.797 | 6.400 | 6.653 |
| 12 | 9.409 | 9.800 | 6.370 | 6.504 |
| 13 | 9.346 | 9.765 | 6.352 | 6.524 |
| 14 | 9.341 | 9.818 | 6.386 | 6.566 |
| 15 | 9.404 | 9.797 | 6.387 | 6.551 |
| 16 | 9.316 | 9.838 | 6.432 | 6.425 |
| 17 | 9.353 | 9.720 | 6.460 | 6.505 |
| 18 | 9.384 | 9.699 | 6.410 | 6.481 |
| 19 | 9.385 | 9.689 | 6.401 | 6.418 |
| 20 | 9.427 | 9.766 | 6.354 | 6.483 |

## Per-epoch longer-baseline times

| Epoch | Longer CURE-Max, seed 0 | Longer CURE-Max, seed 1 |
|---|---:|---:|
| 1 | 0.169 | 0.168 |
| 2 | 4.985 | 5.027 |
| 3 | 5.056 | 5.123 |
| 4 | 5.082 | 5.077 |
| 5 | 5.028 | 5.074 |
| 6 | 5.008 | 5.025 |
| 7 | 4.990 | 5.099 |
| 8 | 5.026 | 5.052 |
| 9 | 5.005 | 5.108 |
| 10 | 5.075 | 5.145 |
| 11 | 5.042 | 5.140 |
| 12 | 5.081 | 5.047 |
| 13 | 5.054 | 5.076 |
| 14 | 5.056 | 5.075 |
| 15 | 5.044 | 5.016 |
| 16 | 5.045 | 5.043 |
| 17 | 4.984 | 5.028 |
| 18 | 5.105 | 5.040 |
| 19 | 5.209 | 5.152 |
| 20 | 4.959 | 5.033 |
| 21 | 4.997 | 5.087 |
| 22 | 4.968 | 5.027 |
| 23 | 5.033 | 5.054 |
| 24 | 5.030 | 5.031 |
| 25 | 4.971 | 5.025 |
| 26 | 5.030 | 5.058 |
| 27 | 4.994 | 5.057 |
| 28 | 5.152 | 5.303 |
| 29 | 4.924 | 5.099 |
| 30 | 4.914 | 5.165 |
| 31 | 5.027 | 5.331 |
| 32 | 4.975 | 5.134 |
| 33 | 5.002 | 5.133 |
| 34 | 4.967 | 5.130 |
| 35 | 4.995 | 5.071 |
| 36 | 4.990 | 5.173 |
| 37 | 5.000 | 4.972 |
| 38 | 4.995 | 5.000 |


In [ ]:
from IPython.display import display, Markdown

all_c_rows = baseline_runs + c_runs
keys = ["clean", "linf", "l2", "union", "pgd_linf", "pgd_l2", "pgd_union"]
lines = ["| Method | " + " | ".join(keys) + " | Total s |", "|---|" + "---:|" * 8]
for name in dict.fromkeys((row["name"] for row in all_c_rows)):
    rr = [row for row in all_c_rows if row["name"] == name]
    values = [sum((row["metrics"][k] for row in rr)) / len(rr) for k in keys]
    total = sum(
        (row.get("total_training_seconds", 20 * row["mean_epoch_seconds"]) for row in rr)
    ) / len(rr)
    lines.append(
        "| " + name + " | " + " | ".join((f"{v:.2f}" for v in values)) + f" | {total:.2f} |"
    )
display(Markdown("\n".join(lines)))
